# Can we predict a warranty claim within a year?
Protocol: `protocol.yaml`. Holdout rows fixed by `holdout.csv`; 5-fold stratified CV on training rows only; choose by mean CV ROC AUC; score the chosen model once on the holdout.

Features: `channel`, `device_age_years`, `price_eur`, `daily_hours`. `device_serial` is excluded (identifier). All four are recorded at sale/registration, before the outcome window. `daily_hours` is reported at registration; I assume that is at or near sale time.

In [ ]:
import json
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==7000 and len(hold_idx)==1400 and len(set(hold_idx))==1400
TARGET='claim_within_year'
FEATURES=['channel','device_age_years','price_eur','daily_hours']
CAT=['channel']; NUM=[f for f in FEATURES if f not in CAT]
is_hold = np.zeros(len(df),bool); is_hold[hold_idx]=True
train = df[~is_hold]; hold = df[is_hold]
train_idx = np.flatnonzero(~is_hold)
X_tr,y_tr = train[FEATURES],train[TARGET]
X_ho,y_ho = hold[FEATURES],hold[TARGET]
print(len(train),len(hold),y_tr.mean(),y_ho.mean())

## Candidates (named before any scoring)
All preprocessing sits inside the pipeline, so it is fitted within each CV fold.

In [ ]:
def prep(scale):
    num = [('imp',SimpleImputer(strategy='median'))]+([('sc',StandardScaler())] if scale else [])
    return ColumnTransformer([('num',Pipeline(num),NUM),
                              ('cat',OneHotEncoder(handle_unknown='ignore'),CAT)])
candidates = {
 'baseline_prior': Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C1': Pipeline([('p',prep(True)),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'logreg_C0.1': Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'random_forest': Pipeline([('p',prep(False)),('m',RandomForestClassifier(n_estimators=300,min_samples_leaf=20,random_state=0))]),
 'hist_gbm': Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

## Cross-validation on training rows

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
cv_scores = {n: cross_val_score(m, X_tr, y_tr, cv=cv, scoring='roc_auc').tolist() for n,m in candidates.items()}
summary = pd.Series({n:np.mean(s) for n,s in cv_scores.items()}).sort_values(ascending=False)
print(summary)
chosen = summary.index[0]; print('chosen:',chosen)

## Refit chosen candidate on all training rows; score once on holdout

In [ ]:
final = candidates[chosen].fit(X_tr,y_tr)
holdout_auc = roc_auc_score(y_ho, final.predict_proba(X_ho)[:,1])
print(f'Holdout ROC AUC ({chosen}): {holdout_auc:.4f}')

In [ ]:
cv_folds = [[train_idx[tr].tolist(), train_idx[te].tolist()] for tr,te in cv.split(X_tr,y_tr)]
evidence = {'features':FEATURES,'candidates':list(candidates),'cv_folds':cv_folds,
            'cv_scores':cv_scores,'chosen':chosen,'holdout_rows':hold_idx.tolist(),
            'holdout_scores':{chosen:holdout_auc}}
json.dump(evidence, open('evidence.json','w'))